# V6.5 — Camber Compose QA Launcher

Notebook này nằm **giữa**:

`V6.5 External Expansion Staging / QA` → **Compose QA** → `YOLO11n@640 Data Ablation Train`

Mục tiêu:

- auto-discover artifact `v6p5_ext_<hash>` đã được staging lên Camber;
- tải `freeze.json`, QA ZIP và overlay TAR;
- kiểm tra hard gate chống leakage / sai taxonomy;
- hiển thị review manifest + contact sheet;
- resolve `KEEP / DROP`;
- đóng gói **synthetic TRAIN đã được duyệt** thành archive riêng;
- tạo `v6p5_compose_<hash>.index.json`, freeze, resolved review manifest, QA summary;
- upload vào `stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/compose_qa/<compose_tag>/`.

Sau khi thành công, notebook train hiện tại sẽ tự thấy `v6p5_compose_*`.

## Review policy

Mặc định `AUTO_APPROVE_RECOMMENDED = False`.

Lần chạy đầu notebook sẽ dừng nếu còn decision trống. Xem contact sheets, điền `MANUAL_DECISIONS`, hoặc sau khi đã review thì bật `AUTO_APPROVE_RECOMMENDED = True`, rồi Run All lại.


In [ ]:
# ==============================================================================
# 0. CONFIG
# ==============================================================================
from pathlib import Path
import os, sys, re, io, json, hashlib, shutil, subprocess, zipfile, tarfile
import pandas as pd
import numpy as np

STAGE_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage"
COMPOSE_ROOT = f"{STAGE_ROOT}/compose_qa"

# None = auto-discover nếu chỉ có đúng 1 v6p5_ext_<12hex>
STAGE_TAG_OVERRIDE = None

# SAFE DEFAULT: bật True chỉ sau khi đã xem QA/contact sheets.
AUTO_APPROVE_RECOMMENDED = False

# Override từng review_id nếu cần.
# MANUAL_DECISIONS = {"V65R000001": "KEEP", "V65R000002": "DROP"}
MANUAL_DECISIONS = {}

# pHash là near-duplicate candidate. Mặc định vẫn chặn để review.
ALLOW_PHASH_CANDIDATES = False
ALLOW_EXACT_DUPLICATES = False

MAX_CONTACT_SHEETS = 24
UPLOAD_CAMBER = True
ALLOW_EXISTING_COMPOSE_TAG = False

WORK = Path("/kaggle/working/v6p5_compose_qa")
CACHE = WORK / "cache"
EXTRACT = WORK / "qa_extract"
OUT = WORK / "out"
for p in [WORK, CACHE, EXTRACT, OUT]:
    p.mkdir(parents=True, exist_ok=True)

print("WORK:", WORK)
print("STAGE_ROOT:", STAGE_ROOT)
print("COMPOSE_ROOT:", COMPOSE_ROOT)


In [ ]:
# ==============================================================================
# 1. CAMBER AUTH + CLI
# ==============================================================================
key = os.environ.get("CAMBER_API_KEY")
if not key:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("CAMBER_API_KEY")
    except Exception:
        key = None

if not key:
    raise RuntimeError("Bật Kaggle Secret CAMBER_API_KEY trước rồi chạy lại notebook.")

os.environ["CAMBER_API_KEY"] = key
camber_bin = Path.home() / ".camber/bin/camber"

if shutil.which("camber") is None and not camber_bin.exists():
    subprocess.check_call(
        "curl -sL https://cli.cambercloud.com/install-v2.sh | bash",
        shell=True,
    )

os.environ["PATH"] = (
    f"{Path.home()}/.camber/bin:{Path.home()}/.local/bin:"
    + os.environ.get("PATH", "")
)
CAMBER = shutil.which("camber") or str(camber_bin)
if not Path(CAMBER).exists():
    raise RuntimeError(f"Camber CLI not found: {CAMBER}")
print("CAMBER:", CAMBER)


In [ ]:
# ==============================================================================
# 2. HELPERS
# ==============================================================================
def run(cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True, env=os.environ.copy())
    if check and r.returncode != 0:
        raise RuntimeError(
            f"Command failed ({r.returncode}): {' '.join(map(str, cmd))}\n"
            f"STDOUT:\n{r.stdout[:8000]}\nSTDERR:\n{r.stderr[:8000]}"
        )
    return r


def stash_ls(remote, recursive=True, check=True):
    cmd = [CAMBER, "stash", "ls"]
    if recursive:
        cmd.append("-r")
    cmd.append(remote)
    return run(cmd, check=check)


def stash_get(remote, local, overwrite=False):
    local = Path(local)
    local.parent.mkdir(parents=True, exist_ok=True)
    if local.exists() and not overwrite:
        return local
    if local.exists():
        if local.is_dir(): shutil.rmtree(local)
        else: local.unlink()
    run([CAMBER, "stash", "cp", remote, str(local)])
    return local


def stash_put(local, remote):
    local = Path(local)
    if not local.exists():
        raise FileNotFoundError(local)
    run([CAMBER, "stash", "cp", str(local), remote])


def sha256_file(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()


def stable_json_hash(obj):
    raw = json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def read_csv_safe(path):
    path = Path(path)
    if not path.exists(): return pd.DataFrame()
    try: return pd.read_csv(path)
    except pd.errors.EmptyDataError: return pd.DataFrame()


def normalize_decision(x):
    x = str(x or "").strip().upper()
    return x if x in {"KEEP", "DROP"} else ""


def tar_find_member(member_names, suffix):
    suffix = str(suffix).replace("\\", "/").lstrip("/")
    hits = [n for n in member_names if n.replace("\\", "/").endswith(suffix)]
    if len(hits) == 1: return hits[0]
    if len(hits) == 0: return None
    raise RuntimeError(f"Ambiguous TAR member suffix {suffix}: {hits[:10]}")


def add_bytes_to_tar(out_tar, arcname, raw, src_info=None):
    info = tarfile.TarInfo(name=str(arcname).replace("\\", "/"))
    info.size = len(raw)
    info.mode = getattr(src_info, "mode", 0o644) or 0o644
    info.mtime = getattr(src_info, "mtime", 0) or 0
    out_tar.addfile(info, io.BytesIO(raw))


def extract_tag_tokens(text, prefix):
    pat = rf"{re.escape(prefix)}[0-9a-fA-F]{{12}}"
    return sorted(set(re.findall(pat, text or "")))

print("Helpers ready.")


In [ ]:
# ==============================================================================
# 3. DISCOVER V6.5 EXTERNAL STAGING TAG
# ==============================================================================
listing = stash_ls(STAGE_ROOT, recursive=True).stdout

if STAGE_TAG_OVERRIDE:
    STAGE_TAG = STAGE_TAG_OVERRIDE.strip()
else:
    candidates = extract_tag_tokens(listing, "v6p5_ext_")
    print("External stage candidates:", candidates)
    if len(candidates) == 0:
        raise RuntimeError(
            "Không tìm thấy v6p5_ext_* dưới STAGE_ROOT.\n"
            "Chạy KAGGLE_V6P5_EXTERNAL_EXPANSION_STAGING_QA trước."
        )
    elif len(candidates) == 1:
        STAGE_TAG = candidates[0]
    else:
        raise RuntimeError(
            "Có nhiều v6p5_ext_* candidates. Set STAGE_TAG_OVERRIDE ở CONFIG:\n"
            + "\n".join(candidates)
        )

STAGE_REMOTE = f"{STAGE_ROOT}/{STAGE_TAG}"
print("SELECTED STAGE_TAG:", STAGE_TAG)
print("STAGE_REMOTE:", STAGE_REMOTE)


In [ ]:
# ==============================================================================
# 4. DOWNLOAD STAGE METADATA + QA + OVERLAY
# ==============================================================================
freeze_remote = f"{STAGE_REMOTE}/{STAGE_TAG}.freeze.json"
qa_remote = f"{STAGE_REMOTE}/{STAGE_TAG}_qa.zip"
overlay_remote = f"{STAGE_REMOTE}/{STAGE_TAG}_overlay.tar.gz"

freeze_local = stash_get(freeze_remote, CACHE / f"{STAGE_TAG}.freeze.json")
qa_local = stash_get(qa_remote, CACHE / f"{STAGE_TAG}_qa.zip")
overlay_local = stash_get(overlay_remote, CACHE / f"{STAGE_TAG}_overlay.tar.gz")
stage_freeze = json.loads(freeze_local.read_text())

print("Downloaded:")
print(" -", freeze_local, f"{freeze_local.stat().st_size/1024**2:.2f} MB")
print(" -", qa_local, f"{qa_local.stat().st_size/1024**2:.2f} MB")
print(" -", overlay_local, f"{overlay_local.stat().st_size/1024**2:.2f} MB")
print(json.dumps(stage_freeze, indent=2, ensure_ascii=False))


In [ ]:
# ==============================================================================
# 5. EXTRACT QA ONLY + LOAD MANIFESTS
# ==============================================================================
if EXTRACT.exists(): shutil.rmtree(EXTRACT)
EXTRACT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(qa_local) as z:
    z.extractall(EXTRACT)

review_path = EXTRACT / "v6p5_candidate_review_manifest.csv"
direct_path = EXTRACT / "direct_overlay_manifest.csv"
synth_path = EXTRACT / "synthetic_foreign_manifest.csv"
exact_path = EXTRACT / "external_vs_internal_exact_duplicates.csv"
phash_path = EXTRACT / "external_vs_internal_phash_candidates.csv"

required = [review_path, direct_path, synth_path]
missing = [p.name for p in required if not p.exists()]
if missing:
    raise RuntimeError("QA ZIP thiếu file bắt buộc: " + ", ".join(missing))

review = read_csv_safe(review_path)
direct = read_csv_safe(direct_path)
synth = read_csv_safe(synth_path)
exact_dup = read_csv_safe(exact_path)
phash_dup = read_csv_safe(phash_path)

print("review rows:", len(review))
print("direct rows:", len(direct))
print("synthetic rows:", len(synth))
print("exact duplicate rows:", len(exact_dup))
print("pHash candidate rows:", len(phash_dup))
display(review.head(10))


In [ ]:
# ==============================================================================
# 6. HARD QA GATES
# ==============================================================================
EXPECTED_NAMES = [
    "insulator", "insulator_broken", "insulator_flashover",
    "bird_nest", "foreign_object", "broken_strand",
]

taxonomy = stage_freeze.get("taxonomy", {})
if isinstance(taxonomy, dict):
    try: got_names = [taxonomy[k] for k in sorted(taxonomy, key=lambda x: int(x))]
    except Exception: got_names = list(taxonomy.values())
else:
    got_names = list(taxonomy)

print("Taxonomy:", got_names)
if got_names != EXPECTED_NAMES:
    raise RuntimeError(f"Taxonomy mismatch. Expected={EXPECTED_NAMES} Got={got_names}")

policy = stage_freeze.get("policy", {})
if policy.get("test_modified") not in (False, None):
    raise RuntimeError("Hard gate failed: stage freeze says test_modified=True")
if policy.get("auto_promotion") not in (False, None):
    raise RuntimeError("Hard gate failed: staging artifact unexpectedly auto-promoted.")

if len(exact_dup) > 0 and not ALLOW_EXACT_DUPLICATES:
    display(exact_dup.head(50))
    raise RuntimeError(f"Hard gate failed: {len(exact_dup)} exact SHA duplicate(s).")

if len(phash_dup) > 0 and not ALLOW_PHASH_CANDIDATES:
    display(phash_dup.head(50))
    raise RuntimeError(
        f"QA gate: còn {len(phash_dup)} pHash near-duplicate candidate(s). "
        "Review rồi mới set ALLOW_PHASH_CANDIDATES=True."
    )

required_review_cols = {
    "review_id", "candidate_type", "overlay_image", "overlay_label", "source",
    "target_split", "mapped_class_names", "recommended_action", "decision",
}
missing_cols = sorted(required_review_cols - set(review.columns))
if missing_cols:
    raise RuntimeError("Review manifest thiếu cột: " + ", ".join(missing_cols))
if review["review_id"].duplicated().any():
    raise RuntimeError("review_id bị duplicate.")
if review["target_split"].astype(str).str.lower().eq("test").any():
    raise RuntimeError("Có candidate target_split=test. Không được compose vào train.")

print("✅ Hard QA gates passed.")


In [ ]:
# ==============================================================================
# 7. QA SUMMARY
# ==============================================================================
summary_cols = [c for c in [
    "candidate_type", "source", "target_split", "mapped_class_names", "recommended_action"
] if c in review.columns]

if len(review):
    display(review.groupby(summary_cols, dropna=False).size().reset_index(name="n").sort_values("n", ascending=False))

print("Stage freeze counts:")
print(json.dumps(stage_freeze.get("counts", {}), indent=2, ensure_ascii=False))

if len(synth) and "foreign_subtype" in synth.columns:
    display(synth["foreign_subtype"].fillna("<NA>").value_counts().rename_axis("foreign_subtype").reset_index(name="n"))

if len(direct):
    gcols = [c for c in ["source", "target_split", "mapped_class_names"] if c in direct.columns]
    display(direct.groupby(gcols, dropna=False).size().reset_index(name="n").sort_values("n", ascending=False))


In [ ]:
# ==============================================================================
# 8. CONTACT SHEET PREVIEW
# ==============================================================================
from IPython.display import display as ipy_display
from PIL import Image

contact_files = sorted([
    p for p in EXTRACT.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    and "contact" in str(p.parent).lower()
])
print("Contact sheets found:", len(contact_files))

for p in contact_files[:MAX_CONTACT_SHEETS]:
    print(p.relative_to(EXTRACT))
    try:
        im = Image.open(p)
        im.thumbnail((1100, 800))
        ipy_display(im)
    except Exception as e:
        print("Could not display:", e)

if len(contact_files) > MAX_CONTACT_SHEETS:
    print(f"... còn {len(contact_files)-MAX_CONTACT_SHEETS} contact sheets.")


In [ ]:
# ==============================================================================
# 9. RESOLVE KEEP / DROP
# ==============================================================================
resolved = review.copy()
resolved["decision"] = resolved["decision"].map(normalize_decision)

for rid, decision in MANUAL_DECISIONS.items():
    decision = normalize_decision(decision)
    if decision not in {"KEEP", "DROP"}:
        raise ValueError(f"Invalid MANUAL_DECISIONS[{rid!r}]={decision!r}")
    mask = resolved["review_id"].astype(str) == str(rid)
    if not mask.any(): raise KeyError(f"review_id not found: {rid}")
    resolved.loc[mask, "decision"] = decision

if AUTO_APPROVE_RECOMMENDED:
    blank = resolved["decision"].eq("")
    train = resolved["target_split"].astype(str).str.lower().eq("train")
    recommended_keep = resolved["recommended_action"].astype(str).str.upper().str.contains("KEEP", na=False)
    resolved.loc[blank & train & recommended_keep, "decision"] = "KEEP"
    resolved.loc[blank & ~(train & recommended_keep), "decision"] = "DROP"

pending_train = resolved[
    resolved["target_split"].astype(str).str.lower().eq("train") & resolved["decision"].eq("")
].copy()

display(
    resolved.assign(decision_display=resolved["decision"].replace("", "PENDING"))
    .groupby(["candidate_type", "target_split", "decision_display"], dropna=False)
    .size().reset_index(name="n")
)

if len(pending_train):
    cols = [c for c in ["review_id", "candidate_type", "source", "mapped_class_names", "subtype", "recommended_action"] if c in pending_train.columns]
    display(pending_train[cols].head(100))
    raise RuntimeError(
        f"Còn {len(pending_train)} TRAIN candidate(s) chưa có decision.\n"
        "Xem contact sheets, điền MANUAL_DECISIONS hoặc sau khi review set AUTO_APPROVE_RECOMMENDED=True, rồi Run All lại."
    )

resolved.loc[resolved["target_split"].astype(str).str.lower().ne("train"), "decision"] = "DROP"
resolved_path = OUT / "v6p5_candidate_review_manifest.resolved.csv"
resolved.to_csv(resolved_path, index=False)
print("✅ Review decisions resolved:", resolved_path)


In [ ]:
# ==============================================================================
# 10. BUILD APPROVED SYNTHETIC TRAIN ARCHIVE — STREAMING / LOW DISK
# ==============================================================================
keep_synth = resolved[
    resolved["candidate_type"].astype(str).eq("synthetic_relation")
    & resolved["target_split"].astype(str).str.lower().eq("train")
    & resolved["decision"].eq("KEEP")
].copy()

print("Approved synthetic rows:", len(keep_synth))
if len(keep_synth) == 0:
    raise RuntimeError("Không có synthetic_relation TRAIN candidate nào được KEEP.")

tmp_synth_tar = OUT / "approved_synthetic_train.tar.gz"

with tarfile.open(overlay_local, "r:gz") as src:
    members = {m.name: m for m in src.getmembers() if m.isfile()}
    names = list(members)
    with tarfile.open(tmp_synth_tar, "w:gz") as dst:
        included = []
        for _, row in keep_synth.iterrows():
            image_base = Path(str(row["overlay_image"])).name
            label_base = Path(str(row["overlay_label"])).name
            image_suffix = f"synthetic_foreign/images/train/{image_base}"
            label_suffix = f"synthetic_foreign/labels/train/{label_base}"
            im_name = tar_find_member(names, image_suffix)
            lb_name = tar_find_member(names, label_suffix)
            if im_name is None: raise RuntimeError(f"Missing synthetic image: {image_suffix}")
            if lb_name is None: raise RuntimeError(f"Missing synthetic label: {label_suffix}")
            im_member, lb_member = members[im_name], members[lb_name]
            im_raw = src.extractfile(im_member).read()
            lb_raw = src.extractfile(lb_member).read()
            out_im = f"v6p5_overlay/synthetic_foreign/images/train/{image_base}"
            out_lb = f"v6p5_overlay/synthetic_foreign/labels/train/{label_base}"
            add_bytes_to_tar(dst, out_im, im_raw, im_member)
            add_bytes_to_tar(dst, out_lb, lb_raw, lb_member)
            included.append({
                "review_id": row["review_id"], "image": out_im, "label": out_lb,
                "image_sha256": hashlib.sha256(im_raw).hexdigest(),
                "label_sha256": hashlib.sha256(lb_raw).hexdigest(),
            })

included_df = pd.DataFrame(included)
if len(included_df) != len(keep_synth):
    raise RuntimeError(f"Approved synthetic mismatch: {len(included_df)} != {len(keep_synth)}")

approved_manifest_path = OUT / "approved_synthetic_manifest.csv"
included_df.to_csv(approved_manifest_path, index=False)
print("Archive:", tmp_synth_tar)
print("Archive MB:", round(tmp_synth_tar.stat().st_size / 1024**2, 3))
print("SHA256:", sha256_file(tmp_synth_tar))
display(included_df.head())


In [ ]:
# ==============================================================================
# 11. FINAL COMPOSE QA + IMMUTABLE TAG
# ==============================================================================
# Train launcher hiện tại lấy direct TRAIN rows trực tiếp từ STAGE_ROOT/mpcd và
# STAGE_ROOT/power_equipment. Compose artifact gate QA/review và cung cấp filtered
# synthetic archive tương thích với launcher đó.

train_rows = resolved[resolved["target_split"].astype(str).str.lower().eq("train")].copy()
kept = train_rows[train_rows["decision"].eq("KEEP")].copy()
dropped = train_rows[train_rows["decision"].eq("DROP")].copy()

review_digest_payload = [{
    "review_id": str(r["review_id"]),
    "candidate_type": str(r["candidate_type"]),
    "source": str(r["source"]),
    "mapped_class_names": str(r["mapped_class_names"]),
    "subtype": str(r.get("subtype", "")),
    "decision": str(r["decision"]),
} for _, r in resolved.sort_values("review_id").iterrows()]

compose_seed = {
    "source_stage_tag": STAGE_TAG,
    "source_overlay_sha256": stage_freeze.get("overlay_sha256"),
    "review_decisions_sha256": stable_json_hash(review_digest_payload),
    "approved_synthetic_archive_sha256": sha256_file(tmp_synth_tar),
    "taxonomy": EXPECTED_NAMES,
}

compose_digest = stable_json_hash(compose_seed)
COMPOSE_TAG = f"v6p5_compose_{compose_digest[:12]}"
COMPOSE_REMOTE = f"{COMPOSE_ROOT}/{COMPOSE_TAG}"

synthetic_final = OUT / f"{COMPOSE_TAG}_synthetic_approved.tar.gz"
shutil.copy2(tmp_synth_tar, synthetic_final)

qa_summary = {
    "compose_tag": COMPOSE_TAG,
    "source_stage_tag": STAGE_TAG,
    "review_total": int(len(resolved)),
    "train_candidates": int(len(train_rows)),
    "train_keep": int(len(kept)),
    "train_drop": int(len(dropped)),
    "approved_synthetic": int(len(keep_synth)),
    "exact_duplicate_candidates": int(len(exact_dup)),
    "phash_candidates": int(len(phash_dup)),
    "allow_phash_candidates": bool(ALLOW_PHASH_CANDIDATES),
    "auto_approve_recommended": bool(AUTO_APPROVE_RECOMMENDED),
}
qa_summary_path = OUT / f"{COMPOSE_TAG}.qa_summary.json"
qa_summary_path.write_text(json.dumps(qa_summary, indent=2, ensure_ascii=False))
print("COMPOSE_TAG:", COMPOSE_TAG)
print(json.dumps(qa_summary, indent=2, ensure_ascii=False))


In [ ]:
# ==============================================================================
# 12. WRITE INDEX + FREEZE
# ==============================================================================
synthetic_remote = f"{COMPOSE_REMOTE}/{synthetic_final.name}"
resolved_remote = f"{COMPOSE_REMOTE}/{COMPOSE_TAG}.review.resolved.csv"
approved_manifest_remote = f"{COMPOSE_REMOTE}/{COMPOSE_TAG}.synthetic_manifest.csv"
qa_summary_remote = f"{COMPOSE_REMOTE}/{COMPOSE_TAG}.qa_summary.json"

compose_index = {
    "compose_tag": COMPOSE_TAG,
    "source_stage_tag": STAGE_TAG,
    "source_stage_remote": STAGE_REMOTE,
    # IMPORTANT: train launcher đọc field này.
    "synthetic_archive": synthetic_remote,
    "resolved_review_manifest": resolved_remote,
    "approved_synthetic_manifest": approved_manifest_remote,
    "qa_summary": qa_summary_remote,
    "source_stage_freeze": freeze_remote,
    "source_stage_qa": qa_remote,
    "source_stage_overlay": overlay_remote,
    "direct_archives": {
        "mpcd": f"{STAGE_ROOT}/mpcd/mpcd_broken_strand_stage.tar.gz",
        "power_equipment": f"{STAGE_ROOT}/power_equipment/power_equipment_stage.tar.gz",
    },
}

index_path = OUT / f"{COMPOSE_TAG}.index.json"
index_path.write_text(json.dumps(compose_index, indent=2, ensure_ascii=False))

compose_freeze = {
    "tag": COMPOSE_TAG,
    "type": "v6p5_compose_qa",
    "source_stage_tag": STAGE_TAG,
    "source_stage_freeze": stage_freeze,
    "taxonomy": {i: n for i, n in enumerate(EXPECTED_NAMES)},
    "counts": qa_summary,
    "policy": {
        "test_modified": False,
        "val_modified": False,
        "external_eval_in_training": False,
        "synthetic_train_only": True,
        "review_resolved": True,
        "auto_approve_recommended": bool(AUTO_APPROVE_RECOMMENDED),
        "phash_candidates_explicitly_allowed": bool(ALLOW_PHASH_CANDIDATES),
    },
    "artifacts": {
        "synthetic_archive": {"remote": synthetic_remote, "sha256": sha256_file(synthetic_final)},
        "resolved_review_manifest": {"remote": resolved_remote, "sha256": sha256_file(resolved_path)},
        "approved_synthetic_manifest": {"remote": approved_manifest_remote, "sha256": sha256_file(approved_manifest_path)},
        "qa_summary": {"remote": qa_summary_remote, "sha256": sha256_file(qa_summary_path)},
        "index": {"remote": f"{COMPOSE_REMOTE}/{index_path.name}", "sha256": sha256_file(index_path)},
    },
    "compose_seed": compose_seed,
}

freeze_out = OUT / f"{COMPOSE_TAG}.freeze.json"
freeze_out.write_text(json.dumps(compose_freeze, indent=2, ensure_ascii=False))
print(json.dumps(compose_index, indent=2, ensure_ascii=False))
print("Freeze:", freeze_out)


In [ ]:
# ==============================================================================
# 13. VERIFY LOCAL COMPOSE ARTIFACTS
# ==============================================================================
with tarfile.open(synthetic_final, "r:gz") as t:
    file_names = [m.name for m in t.getmembers() if m.isfile()]

img_names = sorted([n for n in file_names if "synthetic_foreign/images/train/" in n and Path(n).suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}])
lbl_names = sorted([n for n in file_names if "synthetic_foreign/labels/train/" in n and n.lower().endswith(".txt")])
img_stems = {Path(n).stem for n in img_names}
lbl_stems = {Path(n).stem for n in lbl_names}

print("Synthetic images:", len(img_names))
print("Synthetic labels:", len(lbl_names))
if img_stems - lbl_stems: raise RuntimeError(f"Missing labels: {sorted(img_stems-lbl_stems)[:20]}")
if lbl_stems - img_stems: raise RuntimeError(f"Orphan labels: {sorted(lbl_stems-img_stems)[:20]}")
if len(img_names) != len(keep_synth):
    raise RuntimeError(f"Archive count mismatch: {len(img_names)} != {len(keep_synth)}")

with tarfile.open(synthetic_final, "r:gz") as t:
    for m in t.getmembers():
        if not (m.isfile() and m.name.lower().endswith(".txt")): continue
        txt = t.extractfile(m).read().decode("utf-8", errors="replace")
        for ln_no, line in enumerate(txt.splitlines(), 1):
            line = line.strip()
            if not line: continue
            parts = line.split()
            if len(parts) != 5: raise RuntimeError(f"Bad YOLO row {m.name}:{ln_no}: {line}")
            cid = int(float(parts[0])); vals = list(map(float, parts[1:]))
            if not (0 <= cid < len(EXPECTED_NAMES)): raise RuntimeError(f"Bad class id {cid} at {m.name}:{ln_no}")
            if not all(np.isfinite(vals)): raise RuntimeError(f"Non-finite bbox at {m.name}:{ln_no}")
            if not all(0.0 <= v <= 1.0 for v in vals): raise RuntimeError(f"Out-of-bound bbox at {m.name}:{ln_no}: {vals}")

print("✅ Local compose archive integrity passed.")


In [ ]:
# ==============================================================================
# 14. UPLOAD COMPOSE CANDIDATE TO CAMBER
# ==============================================================================
existing = stash_ls(COMPOSE_ROOT, recursive=True, check=False)
existing_tags = extract_tag_tokens(existing.stdout, "v6p5_compose_") if existing.returncode == 0 else []
print("Existing compose tags:", existing_tags)

if COMPOSE_TAG in existing_tags and not ALLOW_EXISTING_COMPOSE_TAG:
    raise RuntimeError(
        f"{COMPOSE_TAG} đã tồn tại trên Camber. Nếu muốn overwrite thật sự, "
        "set ALLOW_EXISTING_COMPOSE_TAG=True."
    )

if UPLOAD_CAMBER:
    uploads = [
        (synthetic_final, synthetic_remote),
        (resolved_path, resolved_remote),
        (approved_manifest_path, approved_manifest_remote),
        (qa_summary_path, qa_summary_remote),
        (index_path, f"{COMPOSE_REMOTE}/{index_path.name}"),
        (freeze_out, f"{COMPOSE_REMOTE}/{freeze_out.name}"),
    ]
    for local, remote in uploads:
        print("UPLOAD:", local.name, "->", remote)
        stash_put(local, remote)
    print("✅ Upload complete.")
else:
    print("UPLOAD_CAMBER=False; local artifacts only.")


In [ ]:
# ==============================================================================
# 15. SERVER-SIDE VERIFY — WHAT TRAIN NOTEBOOK WILL SEE
# ==============================================================================
r = stash_ls(COMPOSE_REMOTE, recursive=True)
print(r.stdout)
server_listing = r.stdout

must_appear = [f"{COMPOSE_TAG}.index.json", f"{COMPOSE_TAG}.freeze.json", synthetic_final.name]
missing_server = [x for x in must_appear if x not in server_listing]
if missing_server:
    raise RuntimeError("Upload verify failed: " + ", ".join(missing_server))

root_listing = stash_ls(COMPOSE_ROOT, recursive=True).stdout
compose_candidates = extract_tag_tokens(root_listing, "v6p5_compose_")
print("Compose candidates visible to train launcher:", compose_candidates)
if COMPOSE_TAG not in compose_candidates:
    raise RuntimeError("Compose tag uploaded nhưng auto-discovery ở COMPOSE_ROOT chưa thấy tag.")

print("\n" + "=" * 78)
print("READY FOR DATA ABLATION TRAIN")
print("=" * 78)
print("COMPOSE_TAG =", COMPOSE_TAG)
print("COMPOSE_REMOTE =", COMPOSE_REMOTE)
print("\nMở KAGGLE_V6P5_CAMBER_YOLO11N_640_DATA_ABLATION_TRAIN")
print("Giữ COMPOSE_TAG_OVERRIDE = None")
print(f"Expected: Compose candidates: ['{COMPOSE_TAG}']")
print("=" * 78)


## Expected final state

Nếu cell cuối hiện:

```text
READY FOR DATA ABLATION TRAIN
Compose candidates: ['v6p5_compose_xxxxxxxxxxxx']
```

thì quay lại notebook train và Run All.

Train notebook hiện tại sẽ reconstruct **V6.2E immutable** cho TRAIN + frozen VAL, thêm direct V6.5 overlays, đọc `<compose_tag>.index.json`, lấy `synthetic_archive` đã được Compose QA lọc, rồi chạy YOLO11n @ 640 / 50 epochs.

Đây vẫn là **data ablation**, chưa phải final V6.5-vs-V6.4 benchmark.
